# Chest X-Ray Multi-Modal System - Colab Runner

DSAI 413 Assignment 2. Runs end-to-end on a **T4 GPU** in Colab.

Before running anything below:
- `Runtime > Change runtime type > Hardware accelerator = T4 GPU`
- In Colab Secrets (key icon in left sidebar), set:
  - `HF_TOKEN` (Hugging Face read token for MedGemma + ColPali)
  - `KAGGLE_USERNAME` and `KAGGLE_KEY` (only needed if downloading from Kaggle; skip if using Drive copy)
- Mount your Drive so the cached MIMIC-CXR copy at `/content/drive/MyDrive/dsai413-data` is reused (no 16 GB re-download).

## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
# Optional: a pre-extracted MIMIC-CXR copy in Drive. If absent, cell 12 will
# fall back to kagglehub download (requires KAGGLE_USERNAME + KAGGLE_KEY in
# Colab Secrets).
DRIVE_DATA = "/content/drive/MyDrive/assignment-2"
if os.path.isdir(DRIVE_DATA):
    print("Drive dataset root:", DRIVE_DATA)
    !ls -la {DRIVE_DATA} | head -20
else:
    DRIVE_DATA = None
    print("No Drive dataset copy found; will download from Kaggle in cell 12.")


Mounted at /content/drive
Drive dataset root: /content/drive/MyDrive/assignment-2
total 0


## 2. Clone the repo

In [3]:
import os
from google.colab import userdata

# Private repo: clone via a GitHub PAT stored in Colab Secrets (key icon).
#   Secret name: GH_TOKEN   (Personal Access Token with `repo` scope)
# If the repo is public, GH_TOKEN can be left unset.
try:
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    GH_TOKEN = None

REPO_OWNER = "ustafaa"
REPO_NAME = "assignment-2"
REPO_DIR = f"/content/{REPO_NAME}"

if GH_TOKEN:
    clone_url = f"https://{GH_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    clone_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not os.path.exists(REPO_DIR):
    !git clone {clone_url} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}


Cloning into '/content/assignment-2'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 195 (delta 103), reused 195 (delta 103), pack-reused 0 (from 0)
Receiving objects: 100% (195/195), 89.50 KiB | 14.92 MiB/s, done.
Resolving deltas: 100% (103/103), done.
/content/assignment-2


## 3. Install dependencies

`colpali-engine` from GitHub source (PyPI lags transformers-5.x fixes).

In [4]:
!pip install -q -r requirements.txt
!pip install -q -U "git+https://github.com/illuin-tech/colpali"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 119.9 MB/s eta 0:00:0000:010:01


## 4. Authentication & env

HF token (required) and Kaggle creds (optional - only used if `--dataset-path` is NOT set in cell 6).

In [7]:
import os
from google.colab import userdata

# HF_TOKEN must be set in Colab Secrets (key icon, left sidebar).
# Required to download the gated MedGemma + ColPali weights.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle:", os.environ["KAGGLE_USERNAME"])
except Exception:
    print("Kaggle creds not set (OK if using Drive copy).")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))


Kaggle creds not set (OK if using Drive copy).
HF_TOKEN set: True


## 5. Verify GPU

In [8]:
import torch
assert torch.cuda.is_available(), "No CUDA - switch to T4 GPU runtime first."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


## 6. Build the 400-sample manifests from the Drive copy

`--dataset-path` points at the pre-extracted Drive folder, so kagglehub is bypassed entirely (no 16 GB download). Takes ~2 min.

In [11]:
import os

if not os.path.exists("data/sample/manifest_test.csv"):
    if DRIVE_DATA:
        !python -u data/download.py --dataset-path "{DRIVE_DATA}"
    else:
        # kagglehub path: needs KAGGLE_USERNAME + KAGGLE_KEY in Colab Secrets.
        # First run downloads ~16 GB into Colab's ephemeral storage (~10-20 min).
        # The 400-sample subset is then cached to Drive via cell 19 for next session.
        assert os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"), \
            "Set KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets (key icon) and rerun the auth cell."
        !python -u data/download.py
else:
    print("manifests already present in data/sample/, skipping download.")


2026-05-16 20:09:05,162 | INFO | using local dataset path: /content/drive/MyDrive/assignment-2
2026-05-16 20:09:05,163 | ERROR | Could not locate train CSV (expected mimic_cxr_aug_train.csv).
2026-05-16 20:09:05,163 | ERROR | Run with --inspect to see the dataset layout.


## 7. (Optional) Build the QA dataset

Skip if `data/sample/qa_dataset.json` already exists in the repo.

In [10]:
import os
if not os.path.exists("data/sample/qa_dataset.json"):
    !python -u data/build_qa_dataset.py
else:
    print("qa_dataset.json already exists, skipping.")

2026-05-16 20:08:03,449 | INFO | manifest : /content/assignment-2/data/sample/manifest_index.csv
2026-05-16 20:08:03,449 | INFO | output   : /content/assignment-2/data/sample/qa_dataset.json
Traceback (most recent call last):
  File "/content/assignment-2/data/build_qa_dataset.py", line 206, in <module>
    main()
  File "/content/assignment-2/data/build_qa_dataset.py", line 131, in main
    df = pd.read_csv(manifest)
         ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.en

## 8. (Optional) Run evaluation

Produces `results/comparison.json` + `results/comparison.md`. Full run ~50 min on T4.

In [ ]:
# Quick smoke (~5 min):
# !python -u -m src.eval --limit-report 5 --limit-qa 5

# Full run (~50 min):
!python -u -m src.eval

In [ ]:
!cat results/comparison.md

## 9. (Optional) Cache built indexes back to Drive

Saves the slow-to-build indexes to Drive so next session skips Phase 4's 12-min ColPali embed.

In [ ]:
DRIVE_CACHE = "/content/drive/MyDrive/dsai413-cache"
import os
os.makedirs(DRIVE_CACHE, exist_ok=True)
# Top-level manifests (so cell 12's guard can skip download on next session)
!cp -f data/sample/manifest.csv {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/manifest_index.csv {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/manifest_test.csv {DRIVE_CACHE}/ 2>/dev/null || true
# Built indexes
!cp -r data/sample/colpali_index {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/clip_index.faiss {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/clip_index.manifest.csv {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/text_index.faiss {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/text_index.manifest.csv {DRIVE_CACHE}/ 2>/dev/null || true
!cp -f data/sample/qa_dataset.json {DRIVE_CACHE}/ 2>/dev/null || true
!ls -la {DRIVE_CACHE}


## 10. (Optional) Restore cached indexes from Drive on a fresh session

Run this after cell 6 (manifests) but before cell 8 (eval) on a fresh runtime, to skip index rebuilding.

In [ ]:
DRIVE_CACHE = "/content/drive/MyDrive/dsai413-cache"
!mkdir -p data/sample
# Restore top-level manifests first so cell 12's guard skips download.
!cp -f {DRIVE_CACHE}/manifest.csv data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/manifest_index.csv data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/manifest_test.csv data/sample/ 2>/dev/null || true
# Restore built indexes
!cp -r {DRIVE_CACHE}/colpali_index data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/clip_index.faiss data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/clip_index.manifest.csv data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/text_index.faiss data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/text_index.manifest.csv data/sample/ 2>/dev/null || true
!cp -f {DRIVE_CACHE}/qa_dataset.json data/sample/ 2>/dev/null || true
!ls -la data/sample/


## 11. Launch the Gradio demo

Prints a public `gradio.live` URL (share=True). First click per model loads weights (~60 s MedGemma, ~15 s ColPali). Keep cell running to keep the demo alive.

In [ ]:
!python -u app/app.py